## Idea for testing correlating features:

Binary data have no variance. Transform perhaps?
 - no

Create more binary data for ex. age. Threshold <65
BMI thr. 25
DON'T REPLACE the features, but ADD them. (for decision tree)
Does this change the performance of the models? nah, we aint doin that

Collinarity between features. 
 - yes, a lot! But it's a giant dataset so it is expected.

Correlation between singled out features and the output.
Pearson Correlation. 
 - nothing of value

Heatmap and correlation matrix for visuals.
 - also unnecessary.

Find explanations of the data. For instance: Is GenHealth a function of PhysHealth and MenHealth? Where to find?
 - All features do not inherently exist as functions of eachother. 
 - seen https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf




In [1]:
import pandas as pd
from pandas import read_csv
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. Load dataset
file_path = 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv'
df = read_csv(file_path)

# 2. Separate target and predictor features
target_col = 'Diabetes_binary'
X = getattr(df, 'drop')(columns=[target_col])
y = df[target_col]

# 3. Create fixed 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Fit baseline model on ALL features
baseline_model = DecisionTreeClassifier(max_depth=5, random_state=42)
getattr(baseline_model, 'fit')(X_train, y_train)

# Measure baseline test set accuracy
baseline_preds = getattr(baseline_model, 'predict')(X_test)
baseline_acc = accuracy_score(y_test, baseline_preds)

# 5. Run LOFO loop across all 21 features
lofo_results = []

for col in X.columns:
  # Remove single feature col from training and test sets
  X_tr_sub = getattr(X_train, 'drop')(columns=[col])
  X_te_sub = getattr(X_test, 'drop')(columns=[col])

  # Retrain model on remaining 20 features
  sub_model = DecisionTreeClassifier(max_depth=5, random_state=42)
  getattr(sub_model, 'fit')(X_tr_sub, y_train)

  # Measure accuracy without feature col
  sub_preds = getattr(sub_model, 'predict')(X_te_sub)
  sub_acc = accuracy_score(y_test, sub_preds)

  # Drop in performance
  acc_drop = baseline_acc - sub_acc

  lofo_results.append({
      'Feature': col,
      'Accuracy_Without_Feature': sub_acc,
      'LOFO_Importance_Drop': acc_drop,
  })

# 6. Display LOFO table sorted by importance drop
lofo_df = pd.DataFrame(lofo_results)
sorted_lofo = getattr(lofo_df, 'sort_values')(
    by='LOFO_Importance_Drop', ascending=False
)
display(sorted_lofo)

,Feature,Accuracy_Without_Feature,LOFO_Importance_Drop
13,GenHlth,0.712780,0.022420
3,BMI,0.723531,0.011670
0,HighBP,0.726926,0.008275
18,Age,0.729047,0.006153
1,HighChol,0.733715,0.001485
12,NoDocbcCost,0.735201,0.000000
19,Education,0.735201,0.000000
17,Sex,0.735201,0.000000
16,DiffWalk,0.735201,0.000000
15,PhysHlth,0.735201,0.000000
